# CookMatch — Colab Full Run

## Run order

1. **Runtime → Restart session** (if you re-ran cell 1 before and hit errors)
2. Run **cell 1** (downloads + syncs latest code from GitHub)
3. Run **cells 2 → 6** in this tab

Cell 1 always pulls fresh Python files from GitHub `main`, even if git clone is cached.

Flow: setup → Kaggle auth → load data → train → recommend → ablation.

In [13]:
# 1) Bootstrap latest code from GitHub (run once per session)
import os

os.chdir("/content")

# Fetch bootstrap module from GitHub main FIRST (not from a stale clone)
import urllib.request

_RAW = "https://raw.githubusercontent.com/YUV3571/cookmatch-recipe-recommender/main/colab_init.py"
urllib.request.urlretrieve(_RAW, "/content/_cookmatch_colab_init.py")

import importlib.util

_spec = importlib.util.spec_from_file_location("colab_init", "/content/_cookmatch_colab_init.py")
ci = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(ci)

REPO_DIR = ci.bootstrap_repo()
ci.verify_eval_stack()

print("Code OK: latest eval fixes loaded")
print("GitHub ↔ Colab OK | repo:", REPO_DIR)
print("Next: run cells 2 → 6")
!ls {REPO_DIR}/src/data/

Connected via git clone


AssertionError: 

In [ ]:
# 2) Install dependencies
!pip install -q kagglehub pandas numpy scipy pyarrow

In [ ]:
# 3) Kaggle authentication
import os

# Option A (recommended): Kaggle API token
# Kaggle → Settings → API → Create New Token → copy KGAT_... value
os.environ["KAGGLE_API_TOKEN"] = "KGAT_your_token_here"  # paste your token

# Option B: upload kaggle.json instead (comment out Option A first)
# from google.colab import files
# uploaded = files.upload()
# !mkdir -p ~/.kaggle
# !mv kaggle.json ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json

assert os.environ.get("KAGGLE_API_TOKEN", "").startswith("KGAT_"), "Set a valid KAGGLE_API_TOKEN"
print("Kaggle token configured.")

In [ ]:
# 4) Load dataset
import os
import sys
import time

REPO_DIR = "/content/cookmatch-recipe-recommender"
import colab_init
colab_init.bind(REPO_DIR)

# --- RUN MODE ---
FULL_CATALOG = True      # False = quick 5000-row smoke test
FAST_ABLATION = True     # 30k eval catalog + 100 users (~5–15 min). False = hours on 231k
EVAL_CATALOG_SIZE = 30_000
USER_SAMPLE = 100 if FAST_ABLATION else 500
RECIPE_LIMIT = None if FULL_CATALOG else 5000
# ----------------

import kagglehub
from src.data.loader import (
    build_eval_recipe_catalog,
    get_dataset_path,
    load_interaction_split,
    load_recipes,
)

dataset_path = get_dataset_path()
print("Dataset path:", dataset_path)

t0 = time.time()
recipes = load_recipes(
    nrows=RECIPE_LIMIT,
    columns=["id", "name", "ingredients", "minutes", "tags"],
)
train = load_interaction_split("train")
validation = load_interaction_split("validation")

recipes_eval = build_eval_recipe_catalog(
    recipes, train, validation, max_recipes=EVAL_CATALOG_SIZE
)
recipes_train = recipes_eval if FAST_ABLATION else recipes

print(f"recipes loaded: {len(recipes)} ({'FULL' if FULL_CATALOG else 'SAMPLE'})")
print(f"train/eval catalog: {len(recipes_train)} recipes")
print(f"train interactions: {len(train)}")
print(f"validation rows: {len(validation)}")
print(f"ablation users: {USER_SAMPLE}")
print(f"load time: {time.time() - t0:.1f}s")

In [ ]:
# 5) Train Stage 3 recommender + demo recommendations
import colab_init
colab_init.bind("/content/cookmatch-recipe-recommender")

from src.models.user_profile import UserProfile
from src.models.session_context import SessionContext
from src.recommend.stage3 import Stage3Recommender

recipes_train = globals().get("recipes_train", recipes)

from src.data.loader import filter_interactions_to_catalog

train_catalog = filter_interactions_to_catalog(train, recipes_train)
recommender = Stage3Recommender().fit(recipes_train, train_catalog)
print(f"trained on {len(recipes_train)} recipes, {len(train_catalog)} interactions")

profile = UserProfile(diet="vegan", allergens=["nuts", "dairy", "gluten"])
context = SessionContext(pantry=["tomato", "pasta", "garlic"], max_minutes=30, meal_intent="main")
known_user = int(train["user_id"].iloc[0])

recs = recommender.recommend(profile, context, user_id=known_user, top_n=5)
for rec in recs:
    print(f"{rec.final_score:.3f} | {rec.name}")
    print(f"  why: {rec.explanation}")

In [ ]:
# 6) Ablation eval (reuses cell 5 model)
import colab_init
colab_init.bind("/content/cookmatch-recipe-recommender")

import pandas as pd
import time

from src.eval.offline_eval import run_ablation

recipes_train = globals().get("recipes_train", recipes)
USER_SAMPLE = globals().get("USER_SAMPLE", 100)
TOP_K = 10

assert "recommender" in globals(), "Run cell 5 first"

print(f"Running ablation on {len(recipes_train)} recipes, {USER_SAMPLE} users...")
t0 = time.time()

results = run_ablation(
    recipes=recipes_train,
    train_interactions=train,
    held_out=validation,
    user_sample_size=USER_SAMPLE,
    k=TOP_K,
    stage3=recommender,
)

print(f"Ablation done in {(time.time()-t0)/60:.1f} min")
display(results.sort_values(by=f"hit_rate@{TOP_K}", ascending=False))
results.to_csv("ablation_results.csv", index=False)
print("Saved ablation_results.csv")

## Full catalog run

Uncomment below only if you accept long runtimes:

```python
recipes_full = load_recipes(columns=["id", "name", "ingredients", "minutes", "tags"])
recommender = Stage3Recommender().fit(recipes_full, train)
results_full = run_ablation(recipes=recipes_full, train_interactions=train, held_out=validation, user_sample_size=500, k=10)
results_full.to_csv("ablation_full.csv", index=False)
```